# 3.0 Prepare ReID data (RoundaboutHD)
Builds a clean, **ID-disjoint** ReID split from the crops in `RoundaboutHD_Reid_subset/` (see notebook 2 for why the official
split can't be used as is). Output: `data/processed/reid/crops.csv` – one row per crop, images stay in `data/raw` (paths only).

> This notebook explains the steps. The **source of truth is `src/data/make_dataset.py`** (run with `dvc repro -s process_data`);
> both produce the identical file. If you change a rule, change it in the script.

Rules (decided after notebooks 1 and 2):
1. Remove byte-identical duplicate files (keep one copy).
2. Split file id 673 into two vehicles (it merges a semi-truck and a box truck).
3. Drop IDs with < 2 crops.
4. Split **by vehicle ID** into train / val / test. Val and test only use IDs seen in ≥ 2 cameras (needed for cross-camera
   evaluation); single-camera IDs go to train.
5. In val and test: 1 random crop per (ID, camera) is a **query**, all other crops are the **gallery**.
   Evaluation protocol: for each query, ignore gallery crops of the same ID from the same camera.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.data.load import ROOT, RAW, CAMS, load_reid_index, add_image_info, link_reid

SEED = 42
TEST_FRAC, VAL_FRAC = 1 / 3, 1 / 6   # of the multi-camera IDs
OUT = ROOT / "data" / "processed" / "reid"
rng = np.random.default_rng(SEED)

r = link_reid(add_image_info(load_reid_index())).rename(columns={"split": "official_split"})
print(len(r), "crops,", r.id.nunique(), "ids")

65522 crops, 511 ids


## 1. Remove exact duplicates

In [2]:
n = len(r)
r = r.sort_values(["md5", "official_split"]).drop_duplicates("md5", keep="first").sort_values("path").reset_index(drop=True)
print(f"removed {n - len(r)} duplicate files")

removed 650 duplicate files


## 2. Fix file id 673

In [3]:
x = r[r.id == 673]
print(x.groupby(["gid", "cam"], dropna=False).size())
new_id = int(r.id.max()) + 1
box_truck_cams = set(x[x.gid == 664].cam)
semi_cams = set(x[x.gid == 469].cam)
assert not box_truck_cams & semi_cams, "cameras overlap - can't split by camera"
r.loc[(r.id == 673) & r.cam.isin(box_truck_cams), "id"] = new_id
print(f"id 673 cams {sorted(semi_cams)} stay 673, cams {sorted(box_truck_cams)} -> new id {new_id}")

gid    cam
469.0  1      126
       2      187
664.0  3       87
       4      104
dtype: int64
id 673 cams [1, 2] stay 673, cams [3, 4] -> new id 955


## 3. Drop IDs with < 2 crops

In [4]:
cnt = r.groupby("id").size()
r = r[r.id.isin(cnt[cnt >= 2].index)]
print("dropped ids:", sorted(cnt[cnt < 2].index), "| remaining ids:", r.id.nunique())

dropped ids: [939] | remaining ids: 511


## 4. Split by vehicle ID

In [5]:
cams_per_id = r.groupby("id").cam.nunique()
multi = cams_per_id[cams_per_id >= 2].index.to_numpy()
multi = rng.permutation(multi)
n_test, n_val = round(len(multi) * TEST_FRAC), round(len(multi) * VAL_FRAC)
split_of = {i: "test" for i in multi[:n_test]} | {i: "val" for i in multi[n_test:n_test + n_val]}
r["split"] = r.id.map(split_of).fillna("train")
print(r.groupby("split").agg(ids=("id", "nunique"), crops=("path", "size")))

       ids  crops
split            
test   102  18360
train  358  36996
val     51   9515


## 5. Query / gallery for val and test

In [6]:
r["role"] = np.where(r.split == "train", "train", "gallery")
ev = r[r.split != "train"]
q_idx = ev.groupby(["id", "cam"]).sample(1, random_state=SEED).index
r.loc[q_idx, "role"] = "query"
print(r.groupby(["split", "role"]).size())

split  role   
test   gallery    18081
       query        279
train  train      36996
val    gallery     9378
       query        137
dtype: int64


## 6. Vehicle attributes (from `Vehicle_statistic.xlsx`, keyed by cam + SCT track_id)

In [7]:
xl = pd.read_excel(RAW / "Vehicle_statistic.xlsx", sheet_name=None)
attr = pd.concat([xl[f"imagesc{c:03d}"].iloc[:, :6].assign(cam=c) for c in CAMS]).rename(columns={"Number": "track_id"})
attr.columns = [c.lower() for c in attr.columns]
r = r.merge(attr[["cam", "track_id", "category", "colour", "make", "model"]], on=["cam", "track_id"], how="left")
print("crops with a category:", f"{r.category.notna().mean():.1%}")
print(r.groupby("split").category.value_counts(normalize=True).unstack().round(2))

crops with a category: 99.9%
category  Commercial Van  Heavy Duty Trucks  People Carriers and MPV  Pick Up  \
split                                                                           
test                0.04               0.03                     0.10     0.12   
train               0.07               0.08                     0.03     0.11   
val                 0.06               0.16                     0.06     0.13   

category  SUV and Offroaders  Small Passenger Vehicles  
split                                                   
test                    0.50                      0.21  
train                   0.46                      0.26  
val                     0.33                      0.26  


## 7. Checks

In [8]:
ids = {s: set(r[r.split == s].id) for s in ["train", "val", "test"]}
assert not ids["train"] & ids["val"] and not ids["train"] & ids["test"] and not ids["val"] & ids["test"], "ID leakage"
assert not r.md5.duplicated().any(), "duplicate files"
for s in ["val", "test"]:
    e = r[r.split == s]
    gal = e[e.role == "gallery"].groupby("id").cam.agg(set)
    q = e[e.role == "query"]
    ok = [bool(gal.get(i, set()) - {c}) for i, c in zip(q.id, q.cam)]
    assert all(ok), f"{s}: {len(ok) - sum(ok)} queries without a cross-camera gallery match"
print("all checks passed")

all checks passed


## 8. Write output

In [9]:
OUT.mkdir(parents=True, exist_ok=True)
cols = ["path", "id", "cam", "frame", "split", "role", "official_split", "track_id", "gid", "w", "h",
        "category", "colour", "make", "model"]
out = r[cols].astype({"track_id": "Int64", "gid": "Int64"}).sort_values(["split", "id", "cam", "frame"])
out.to_csv(OUT / "crops.csv", index=False)
print(OUT / "crops.csv", out.shape)
out.head()

/Users/simon/Developer/DSIA/Multi-Vehicle-ReID/data/processed/reid/crops.csv (64871, 15)


,path,id,cam,frame,split,role,official_split,track_id,gid,w,h,category,colour,make,model
0,data/raw/RoundaboutHD/RoundaboutHD_Reid_subset...,3,1,311,test,gallery,query,22,2,85,41,Small Passenger Vehicles,Black,Mercedes,C Class
23938,data/raw/RoundaboutHD/RoundaboutHD_Reid_subset...,3,1,314,test,gallery,train,22,2,91,44,Small Passenger Vehicles,Black,Mercedes,C Class
1171,data/raw/RoundaboutHD/RoundaboutHD_Reid_subset...,3,1,316,test,gallery,test,22,2,93,43,Small Passenger Vehicles,Black,Mercedes,C Class
23939,data/raw/RoundaboutHD/RoundaboutHD_Reid_subset...,3,1,317,test,gallery,train,22,2,92,45,Small Passenger Vehicles,Black,Mercedes,C Class
23940,data/raw/RoundaboutHD/RoundaboutHD_Reid_subset...,3,1,320,test,gallery,train,22,2,99,46,Small Passenger Vehicles,Black,Mercedes,C Class
